# Tutorial: Power dataset regression with Rich-BLL

This notebook focuses only on the `power` dataset. It keeps the benchmark setup used in the repository, but simplifies the presentation to a single dataset and three methods:

1. `map`
2. `full_ntkapprox`
3. `subsampled_ntkapprox`

In [ ]:
import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import torch
from IPython.display import display

ROOT = Path.cwd().resolve()
if not ((ROOT / "experiments").exists() and (ROOT / "src").exists()):
    fallback = Path("/nfs/home/sergioco/test_code/ntk-predictors")
    if (fallback / "experiments").exists() and (fallback / "src").exists():
        ROOT = fallback
    else:
        raise RuntimeError("Could not locate the repository root.")

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from experiments.backbone import train_backbone_once
from experiments.data import get_dataset
from experiments.metrics import (
    gaussian_cqm_diag,
    gaussian_crps_diag,
    gaussian_nll_diag,
    gaussian_nll_full,
    rmse_metric,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
torch.set_printoptions(precision=4, sci_mode=False)

print({"root": str(ROOT), "device": device})

{'root': '/nfs/home/sergioco/test_code/ntk-predictors', 'device': 'cuda'}


## Configuration

The settings below are the ones used in this notebook run. The default subset size follows the same `k = 0.5 * n_train` rule used in the regression benchmark script.


In [ ]:
NOISE_GRID = np.logspace(-2, 1, 50) / 2

CONFIG = {
    "dataset": "power",
    "seeds": [0, 1, 2],
    "parametrization": "mup",
    "k_ratio": 0.5,
    "noise_grid": NOISE_GRID,
    "reuse_backbone": True,
    "require_backbone": False,
    "backbones_dir": ROOT / "results/backbones",
    "use_network_mean": True,
    "use_empirical_noise_for_ntkapprox": True,
}

{'dataset': 'power',
 'seeds': [0, 1, 2],
 'parametrization': 'mup',
 'k_ratio': 0.5,
 'noise_grid': array([0.005     , 0.00575698, 0.00662856, 0.00763209, 0.00878755,
        0.01011795, 0.01164976, 0.01341348, 0.01544422, 0.0177824 ,
        0.02047458, 0.02357433, 0.02714338, 0.03125276, 0.03598428,
        0.04143214, 0.04770477, 0.05492706, 0.06324276, 0.07281742,
        0.08384165, 0.09653489, 0.11114982, 0.1279774 , 0.14735259,
        0.16966109, 0.195347  , 0.22492163, 0.25897373, 0.29818117,
        0.34332442, 0.39530216, 0.45514909, 0.52405657, 0.60339632,
        0.69474775, 0.79992936, 0.92103498, 1.06047544, 1.22102655,
        1.40588435, 1.61872877, 1.86379686, 2.14596713, 2.47085668,
        2.84493301, 3.27564278, 3.77156003, 4.34255687, 5.        ]),
 'reuse_backbone': True,
 'require_backbone': False,
 'backbones_dir': PosixPath('/nfs/home/sergioco/test_code/ntk-predictors/results/backbones'),
 'use_network_mean': True,
 'use_empirical_noise_for_ntkapprox': True}

In [3]:
def sample_subset(x, y, k=None, seed=0):
    if k is None or k >= x.shape[0]:
        return x, y
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(x.shape[0], generator=g)[:k]
    return x[idx], y[idx]


def gp_inference_feature_space(
    features_train: torch.Tensor,
    y_train: torch.Tensor,
    features_test: torch.Tensor,
    noise_variance: float,
    compute_only_covariance: bool = False,
    covariance: str = "full",
    full_data_size: Optional[int] = None,
):
    U = features_train
    U_test = features_test
    y = y_train.reshape(-1)

    r = U.shape[1]
    eye_r = torch.eye(r, device=U.device, dtype=U.dtype)
    N = U.shape[0] if full_data_size is None else full_data_size
    k = U.shape[0]
    scale = N / k
    pop_matrix = scale * (U.T @ U)

    A = pop_matrix + noise_variance * eye_r
    A_inv_UT_test = torch.linalg.solve(A, U_test.T)

    if covariance == "diag":
        cov = noise_variance * (U_test * A_inv_UT_test.T).sum(dim=1)
    elif covariance == "full":
        cov = noise_variance * (U_test @ A_inv_UT_test)
    else:
        raise ValueError("covariance must be 'diag' or 'full'")

    if compute_only_covariance:
        return cov

    UT = scale * U.T
    A_inv_UT = torch.linalg.solve(A, UT)
    w_mean = A_inv_UT @ y
    mu = U_test @ w_mean
    return mu, cov


class FeatureProjection:
    def __init__(self, net, x_train: torch.Tensor, method: str = "Cholesky, compute A", jitter: float = 1e-6):
        self.net = net
        self.x_train = x_train
        self.method = method
        self.jitter = jitter
        self.features_nngp = self.net.nngp_features(self.x_train)
        self.features_ntk = self.net.ntk_features(self.x_train)
        self._fit_projection()

    def _fit_projection(self) -> None:
        N, r = self.features_nngp.shape
        if N < r:
            raise ValueError(f"Need N >= r for the projection fit. Got N={N}, r={r}.")
        if self.method != "Cholesky, compute A":
            raise ValueError(f"Unsupported projection method: {self.method}")

        gram_nngp = self.features_nngp.T @ self.features_nngp
        gram_nngp = gram_nngp + self.jitter * torch.eye(gram_nngp.shape[0], device=gram_nngp.device)
        L_nngp = torch.linalg.cholesky(gram_nngp)
        X = torch.cholesky_solve(self.features_nngp.T, L_nngp).T
        A = self.features_ntk.T @ X
        M = A.T @ A + torch.eye(self.features_nngp.shape[1], device=A.device)

        self.A = A
        self.M = M
        M = 0.5 * (M + M.T)
        self.L = torch.linalg.cholesky(
            M + self.jitter * torch.eye(M.shape[0], device=M.device),
            upper=False,
        )

    def ntkapprox_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.net.nngp_features(x) @ self.L


def get_features(model, x_fit, x_eval, method, fp=None):
    if method == "map":
        return model.nngp_features(x_eval), None
    if method == "ntkapprox":
        if fp is None:
            fp = FeatureProjection(model, x_fit, method="Cholesky, compute A", jitter=1e-6)
        return fp.ntkapprox_features(x_eval), fp
    raise ValueError(f"Unknown method: {method}")


def sweep_noise_variance(
    model,
    train_data,
    val_data,
    noise_grid,
    subset_seed=0,
    subset_k=None,
    method="ntkapprox",
    nll_mode="diag",
    use_empirical_noise=False,
    full_data_size=None,
    use_network_mean=True,
):
    x_train, y_train = train_data
    x_val, y_val = val_data

    x_sub, y_sub = sample_subset(x_train, y_train, k=subset_k, seed=subset_seed)

    fp = None
    U_train, fp = get_features(model, x_sub, x_sub, method, fp=fp)
    U_val, fp = get_features(model, x_sub, x_val, method, fp=fp)

    def empirical_mse_variance(model_ref, data_ref):
        x_ref, y_ref = data_ref
        model_ref.eval()
        with torch.no_grad():
            mu_ref = model_ref(x_ref).squeeze()
            mse = torch.mean((y_ref.squeeze() - mu_ref) ** 2)
        return float(torch.clamp(mse, min=1e-6).item())

    if use_empirical_noise:
        best_noise = empirical_mse_variance(model, val_data)
        noise_grid = [best_noise]
    else:
        best_noise = None

    best_nll = float("inf")

    for nv in noise_grid:
        nv = float(nv)
        if nll_mode == "diag":
            if use_network_mean:
                var_val = gp_inference_feature_space(
                    U_train,
                    y_sub.squeeze(),
                    U_val,
                    noise_variance=nv,
                    covariance="diag",
                    full_data_size=full_data_size,
                    compute_only_covariance=True,
                )
                mu_val = model(x_val).squeeze()
            else:
                mu_val, var_val = gp_inference_feature_space(
                    U_train,
                    y_sub.squeeze(),
                    U_val,
                    noise_variance=nv,
                    covariance="diag",
                    full_data_size=full_data_size,
                )
            pred_var = var_val + nv
            val_nll = gaussian_nll_diag(y_val.squeeze(), mu_val.squeeze(), pred_var)
        elif nll_mode == "full":
            if use_network_mean:
                cov_val = gp_inference_feature_space(
                    U_train,
                    y_sub.squeeze(),
                    U_val,
                    noise_variance=nv,
                    covariance="full",
                    full_data_size=full_data_size,
                    compute_only_covariance=True,
                )
                mu_val = model(x_val).squeeze()
            else:
                mu_val, cov_val = gp_inference_feature_space(
                    U_train,
                    y_sub.squeeze(),
                    U_val,
                    noise_variance=nv,
                    covariance="full",
                    full_data_size=full_data_size,
                )
            pred_cov = cov_val + nv * torch.eye(cov_val.shape[0], device=cov_val.device, dtype=cov_val.dtype)
            try:
                val_nll = gaussian_nll_full(y_val.squeeze(), mu_val.squeeze(), pred_cov)
            except RuntimeError:
                continue
        else:
            raise ValueError("nll_mode must be 'full' or 'diag'")

        if val_nll < best_nll:
            best_nll = val_nll
            best_noise = nv

    if best_noise is None:
        raise ValueError("All noise values failed during the noise sweep.")

    return best_noise, float(best_nll)


def evaluate_method(
    model,
    train_data,
    test_data,
    noise_variance,
    subset_seed=0,
    subset_k=None,
    method="ntkapprox",
    nll_mode="diag",
    full_data_size=None,
    use_network_mean=True,
):
    x_train, y_train = train_data
    x_test, y_test = test_data

    x_sub, y_sub = sample_subset(x_train, y_train, k=subset_k, seed=subset_seed)

    fp = None
    U_train, fp = get_features(model, x_sub, x_sub, method, fp=fp)
    U_test, fp = get_features(model, x_sub, x_test, method, fp=fp)

    nv = float(noise_variance)

    if nll_mode == "diag":
        if use_network_mean:
            var_test = gp_inference_feature_space(
                U_train,
                y_sub.squeeze(),
                U_test,
                noise_variance=nv,
                covariance="diag",
                full_data_size=full_data_size,
                compute_only_covariance=True,
            )
            mu_test = model(x_test).squeeze()
        else:
            mu_test, var_test = gp_inference_feature_space(
                U_train,
                y_sub.squeeze(),
                U_test,
                noise_variance=nv,
                covariance="diag",
                full_data_size=full_data_size,
            )
        pred_var = var_test + nv
        nll = gaussian_nll_diag(y_test.squeeze(), mu_test.squeeze(), pred_var)
        crps = gaussian_crps_diag(y_test.squeeze(), mu_test.squeeze(), pred_var)
        cqm = gaussian_cqm_diag(y_test.squeeze(), mu_test.squeeze(), pred_var)
        return {
            "rmse": rmse_metric(y_test.squeeze(), mu_test.squeeze()).item(),
            "nll": float(nll.item()),
            "subset_size": x_sub.shape[0],
            "crps": float(crps),
            "cqm": float(cqm),
        }

    if nll_mode == "full":
        if use_network_mean:
            cov_test = gp_inference_feature_space(
                U_train,
                y_sub.squeeze(),
                U_test,
                noise_variance=nv,
                covariance="full",
                full_data_size=full_data_size,
                compute_only_covariance=True,
            )
            mu_test = model(x_test).squeeze()
        else:
            mu_test, cov_test = gp_inference_feature_space(
                U_train,
                y_sub.squeeze(),
                U_test,
                noise_variance=nv,
                covariance="full",
                full_data_size=full_data_size,
            )
        pred_cov = cov_test + nv * torch.eye(cov_test.shape[0], device=cov_test.device, dtype=cov_test.dtype)
        nll = gaussian_nll_full(y_test.squeeze(), mu_test.squeeze(), pred_cov)
        pred_var_diag = torch.diag(cov_test) + nv
        crps = gaussian_crps_diag(y_test.squeeze(), mu_test.squeeze(), pred_var_diag)
        cqm = gaussian_cqm_diag(y_test.squeeze(), mu_test.squeeze(), pred_var_diag)
        return {
            "rmse": rmse_metric(y_test.squeeze(), mu_test.squeeze()).item(),
            "nll": float(nll.item()),
            "subset_size": x_sub.shape[0],
            "crps": float(crps),
            "cqm": float(cqm),
        }

    raise ValueError("nll_mode must be 'full' or 'diag'")

## Dataset and backbone summary

We first inspect the split sizes for `power`, then load the pretrained or newly trained backbone and inspect the dimensions of the two feature spaces used by the method.


In [ ]:
train_ds, val_ds, test_ds, _ = get_dataset(CONFIG["dataset"], seed=CONFIG["seeds"][0])
x_train_probe, _ = train_ds.tensors

overview_df = pd.DataFrame([
    {
        "dataset": CONFIG["dataset"],
        "input_dim": int(x_train_probe.shape[1]),
        "train_size": int(len(train_ds)),
        "val_size": int(len(val_ds)),
        "test_size": int(len(test_ds)),
        "trainval_size": int(len(train_ds) + len(val_ds)),
    }
])
display(overview_df)

bundle0 = train_backbone_once(
    CONFIG["dataset"],
    seed=CONFIG["seeds"][0],
    parametrization=CONFIG["parametrization"],
    reuse_backbone=CONFIG["reuse_backbone"],
    weights_dir=CONFIG["backbones_dir"],
    require_backbone=CONFIG["require_backbone"],
)
model0 = bundle0["model"]
train_X0, _ = bundle0["train"]
probe = train_X0[:8]

backbone_df = pd.DataFrame([
    {
        "hidden_dim": int(model0.hidden_dim),
        "num_hidden_layers": int(model0.num_hidden_layers),
        "trainable_params": int(sum(p.numel() for p in model0.parameters())),
        "last_layer_feature_dim": int(model0.nngp_features(probe).shape[-1]),
        "full_ntk_feature_dim": int(model0.ntk_features(probe).shape[-1]),
        "best_epoch": bundle0["best_epoch"],
        "best_val_rmse": bundle0["val_rmse"],
    }
])
display(backbone_df)

,dataset,input_dim,train_size,val_size,test_size,trainval_size
0,power,4,6888,1723,957,8611


,hidden_dim,num_hidden_layers,trainable_params,last_layer_feature_dim,full_ntk_feature_dim,best_epoch,best_val_rmse
0,50,2,2851,51,2800,3000,4.215229


## Step-by-step method summary

Let `φ_L(x)` denote the last-layer Jacobian features and `φ_E(x)` denote the remaining-parameter Jacobian features.

The `ntkapprox` construction in this notebook is:

1. pick a fitting set `S`
2. compute `φ_L` and `φ_E` on `S`
3. fit a projection from `φ_E` into the span of `φ_L`
4. define projected features `u(x) = φ_L(x) L`
5. run Bayesian linear regression in the `u(x)` feature space

The posterior mean is the deterministic network prediction, and the feature-space posterior provides the uncertainty term.

The subsampled variant uses the same `N / k` rescaling as the main regression benchmark when the projection is fit on a subset.


In [5]:
def training_subset_k(bundle, ratio: float) -> int:
    train_X, _ = bundle["train"]
    hidden_dim = int(getattr(bundle["model"], "hidden_dim", 1))
    k = int(round(ratio * train_X.shape[0]))
    k = max(hidden_dim, k)
    return min(k, int(train_X.shape[0]))


def trainval_subset_k(bundle, k_train: int) -> int:
    train_X, _ = bundle["train"]
    trainval_X, _ = bundle["trainval"]
    hidden_dim = int(getattr(bundle["model"], "hidden_dim", 1))
    ratio = k_train / train_X.shape[0]
    k = int(round(ratio * trainval_X.shape[0]))
    k = max(hidden_dim, k)
    return min(k, int(trainval_X.shape[0]))


def run_config(bundle, *, seed: int, label: str, method: str, subset_k_train: Optional[int], nll_mode: str) -> dict:
    model_train = bundle["model_train"]
    model = bundle["model"]
    train_X, train_Y = bundle["train"]
    trainval_X, trainval_Y = bundle["trainval"]
    x_val, y_val = bundle["val"]
    x_test, y_test = bundle["test"]

    sweep_full_data_size = None
    if method == "ntkapprox" and subset_k_train is not None and subset_k_train < train_X.shape[0]:
        sweep_full_data_size = int(train_X.shape[0])

    use_empirical_noise = method == "ntkapprox" and CONFIG["use_empirical_noise_for_ntkapprox"]

    best_noise, best_val_nll = sweep_noise_variance(
        model_train,
        (train_X, train_Y),
        (x_val, y_val),
        noise_grid=CONFIG["noise_grid"],
        subset_seed=seed,
        subset_k=subset_k_train,
        method=method,
        nll_mode=nll_mode,
        use_empirical_noise=use_empirical_noise,
        full_data_size=sweep_full_data_size,
        use_network_mean=CONFIG["use_network_mean"],
    )

    subset_k_eval = subset_k_train
    eval_full_data_size = None
    if method == "ntkapprox" and subset_k_train is not None and subset_k_train < train_X.shape[0]:
        subset_k_eval = trainval_subset_k(bundle, subset_k_train)
        eval_full_data_size = int(trainval_X.shape[0])

    eval_res = evaluate_method(
        model,
        (trainval_X, trainval_Y),
        (x_test, y_test),
        noise_variance=best_noise,
        subset_seed=seed,
        subset_k=subset_k_eval,
        method=method,
        nll_mode=nll_mode,
        full_data_size=eval_full_data_size,
        use_network_mean=CONFIG["use_network_mean"],
    )

    return {
        "seed": seed,
        "label": label,
        "method": method,
        "nll_mode": nll_mode,
        "noise_variance": float(best_noise),
        "val_nll": float(best_val_nll),
        "test_nll": float(eval_res["nll"]),
    }


def run_seed(seed: int) -> pd.DataFrame:
    bundle = train_backbone_once(
        CONFIG["dataset"],
        seed=seed,
        parametrization=CONFIG["parametrization"],
        reuse_backbone=CONFIG["reuse_backbone"],
        weights_dir=CONFIG["backbones_dir"],
        require_backbone=CONFIG["require_backbone"],
    )
    subset_k = training_subset_k(bundle, CONFIG["k_ratio"])

    configs = [
        {"label": "map", "method": "map", "subset_k_train": None, "nll_mode": "diag"},
        {"label": "full_ntkapprox", "method": "ntkapprox", "subset_k_train": None, "nll_mode": "full"},
        {
            "label": f"subsampled_ntkapprox_{int(100 * CONFIG['k_ratio'])}pct",
            "method": "ntkapprox",
            "subset_k_train": subset_k,
            "nll_mode": "full",
        },
    ]

    rows = [
        run_config(
            bundle,
            seed=seed,
            label=cfg["label"],
            method=cfg["method"],
            subset_k_train=cfg["subset_k_train"],
            nll_mode=cfg["nll_mode"],
        )
        for cfg in configs
    ]
    return pd.DataFrame(rows)

## Run the comparison

This evaluates the three variants on the `power` dataset for the default seed list.


In [6]:
result_frames = []

for seed in CONFIG["seeds"]:
    print(f"Running dataset={CONFIG['dataset']}, seed={seed}")
    result_frames.append(run_seed(seed))

results_df = pd.concat(result_frames, ignore_index=True)
results_df = results_df.sort_values(["seed", "label"]).reset_index(drop=True)
display(results_df[["seed", "label", "test_nll"]])

Running dataset=power, seed=0
Running dataset=power, seed=1
Running dataset=power, seed=2


,seed,label,test_nll
0,0,full_ntkapprox,2.749869
1,0,map,3.137700
2,0,subsampled_ntkapprox_50pct,2.749844
3,1,full_ntkapprox,2.775950
4,1,map,3.231784
5,1,subsampled_ntkapprox_50pct,2.775972
6,2,full_ntkapprox,2.739367
7,2,map,3.126085
8,2,subsampled_ntkapprox_50pct,2.739390


## Aggregate summary

The main quantity of interest here is mean test NLL.


In [7]:
summary_df = (
    results_df.groupby(["label"], as_index=False)
    .agg(
        mean_test_nll=("test_nll", "mean"),
        n_seeds=("seed", "nunique"),
    )
    .sort_values(["label"])
    .reset_index(drop=True)
)
display(summary_df)

,label,mean_test_nll,n_seeds
0,full_ntkapprox,2.755062,3
1,map,3.165190,3
2,subsampled_ntkapprox_50pct,2.755069,3
